In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Queries, Keys, and Values

The recurrent networks of that section keep everything they know
about the past in a state of fixed size, and every prediction is made from
that state. The network has to decide what to write there before it knows
what a later prediction will need. Consider a recurrent language model as
wide as GPT-2 small that has read 1,024 tokens. Its state holds 768
numbers. If the next prediction depends on the 17th token, it must be made
from those 768 numbers, all written before the network could know that the
17th token would matter. Convolutions over sequences
[@Kalchbrenner.Grefenstette.Blunsom.2014] face the same limit, and on
long inputs the fixed-size summary loses information that a later
prediction needs. Can a model keep its whole input and read from it only
the part that each prediction uses?

A database does this. It stores its records as they are and answers each
query by comparing it with the stored keys. An exact lookup, however,
either finds a matching key or returns nothing, and a network trained by
gradient descent needs an answer that changes smoothly with the query.
Consider the scalar database $\mathcal{D}=\{(2,10),(5,20)\}$ of (key,
value) pairs. The exact query $q=5$ returns the value 20, and the query
$q=4$ returns nothing at all, because no key matches. A *soft* query instead scores every key,
for example by $a(q,k)=-(q-k)^2$, and turns the scores into weights that
sum to one. For $q=4$ the scores are $-4$ for key 2 and $-1$ for key 5. The
softmax turns them into the weights $e^{-4}/(e^{-4}+e^{-1}) \approx 0.05$
and $0.95$, so the answer is $0.05 \cdot 10 + 0.95 \cdot 20 \approx 19.5$.
The answer moves smoothly with the query, the same rule works for a
database of any size, and nothing has to be compressed before the query
arrives.

The *attention mechanism* [@Bahdanau.Cho.Bengio.2014] is this soft
lookup, used as a layer of a neural network. This section defines it and
studies it in its oldest form, kernel regression, where the weights come
from a fixed similarity function rather than from learning. The next
section makes the scores learnable.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## Attention as Differentiable Lookup

The soft query of the example used scalars and two stored pairs. In a
network the keys and values are vectors, and there is a pair for every
position of the input, so we now write the same rule in that generality.

Let the query and the keys be vectors $\mathbf{q},\mathbf{k}_i \in
\mathbb{R}^{d}$ and the values $\mathbf{v}_i \in \mathbb{R}^{d_v}$, and let
$\mathcal{D} \stackrel{\textrm{def}}{=} \{(\mathbf{k}_1, \mathbf{v}_1),
\ldots, (\mathbf{k}_m, \mathbf{v}_m)\}$ be a database of $m$ key--value
pairs. The attention output is the weighted sum of the values

$$\textrm{Attention}(\mathbf{q}, \mathcal{D}) \stackrel{\textrm{def}}{=} \sum_{i=1}^m \alpha(\mathbf{q}, \mathbf{k}_i) \mathbf{v}_i,$$

where the scalar weight $\alpha(\mathbf{q},\mathbf{k}_i)$ says how much
value $i$ contributes. In every softmax attention layer of this book the
weights are nonnegative and sum to one, so the output is a convex
combination of the values: it lies inside their convex hull. Exact lookup
is the special case in which one weight is $1$ and the others are $0$. Uniform
weights $1/m$ give the plain average of the values, *average pooling*.

Weights with these two properties come from any scoring function
$a(\mathbf{q}, \mathbf{k})$ by exponentiating the scores, which makes them
positive, and dividing by their sum. This is the softmax operation of
multinomial logistic regression (that section):

$$\alpha(\mathbf{q}, \mathbf{k}_i) = \frac{\exp(a(\mathbf{q}, \mathbf{k}_i))}{\sum_j \exp(a(\mathbf{q}, \mathbf{k}_j))}.$$

The softmax is differentiable, so gradients reach the query, the keys, and
the values. For finite scores and at least two keys its Jacobian is never
the zero matrix, although its entries become very small when one score
dominates the others. Attention can also make hard, nondifferentiable
choices. It must then be trained with reinforcement learning
[@Mnih.Heess.Graves.ea.2014] and is harder to optimize, so this book
uses the differentiable form throughout. the figure works one lookup
through with three keys.

![Soft lookup with three key--value pairs. The query's scores $s_i$ against the keys become softmax weights $\alpha_i$ that sum to one (left), and the output $\mathbf{o}$ is the weighted sum of the values, a point inside the triangle whose corners are the values (right).](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-attention-soft-lookup.svg)

The scoring function has a fixed number of parameters, however many
key--value pairs the database holds. One attention layer therefore serves
inputs of every length, and it reads all of them directly instead of a
compressed summary. What grows instead is memory. A recurrent network
keeps one state, while attention keeps a key and a value for every
position it may need to read.

## Attention Pooling with Fixed Kernels

The weights in the equation need not be learned. Long
before deep learning, statisticians used the same weighted sum with a
weight function chosen by hand, in Nadaraya--Watson kernel regression
[@Nadaraya.1964; @Watson.1964]. On a problem small enough to plot, this
classical setting shows what the weights do. Its limits show why the rest
of the chapter learns them.

### Similarity Kernels

A *kernel* $\alpha(\mathbf{q}, \mathbf{k})$ measures the similarity of a
query and a key, typically as a decreasing function of their distance.
Common choices are

$$
\begin{aligned}
\alpha(\mathbf{q}, \mathbf{k}) & = \exp\left(-\tfrac{1}{2} \|\mathbf{q} - \mathbf{k}\|^2 \right) && \textrm{Gaussian;} \\
\alpha(\mathbf{q}, \mathbf{k}) & = 1 \textrm{ if } \|\mathbf{q} - \mathbf{k}\| \leq 1 && \textrm{boxcar;} \\
\alpha(\mathbf{q}, \mathbf{k}) & = \mathop{\mathrm{max}}\left(0, 1 - \|\mathbf{q} - \mathbf{k}\|\right) && \textrm{triangular,}
\end{aligned}
$$

and the degenerate constant kernel $\alpha(\mathbf{q}, \mathbf{k}) = 1$,
which ignores the query. the figure plots them. Each
can be widened or narrowed by dividing the distance by a *bandwidth*
$\sigma$. The same kernels estimate densities, where they are known as
*Parzen windows* [@parzen1957consistent].

![Four similarity kernels as functions of the query--key distance. Each can be rescaled by a bandwidth.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-attention-kernels.svg)

Normalizing a kernel over the keys, as in the equation but
without the exponential because kernels are already nonnegative, gives the
*Nadaraya--Watson estimator*

$$f(\mathbf{q}) = \sum_i \mathbf{v}_i \frac{\alpha(\mathbf{q}, \mathbf{k}_i)}{\sum_j \alpha(\mathbf{q}, \mathbf{k}_j)}.$$

For scalar regression with observations $(x_i, y_i)$, the keys are the
training inputs $x_i$, the values are the labels $y_i$, and the query is
the input at which we want a prediction. The estimate is a weighted
average of the labels of the training inputs near the query, which is
attention pooling term for term. It needs no training. If the bandwidth
shrinks at a suitable rate as data accumulate, the estimate converges to
the best possible predictor [@mack1982weak].

### Nadaraya--Watson Regression

A one-dimensional regression problem is small enough to plot both the
estimate and every weight behind it, so it shows what the bandwidth does
before any learning is involved. We draw $40$ training inputs uniformly
from $[0, 5)$, label them by $y = 2\sin(x) + x + \epsilon$ with standard
Gaussian noise $\epsilon$, and keep a grid of noise-free points for
comparison.

In [ ]:
n = 40
x_train, _ = torch.sort(torch.rand(n) * 5)
y_train = 2 * torch.sin(x_train) + x_train + torch.randn(n)
x_val = torch.arange(0, 5, 0.1)
y_val = 2 * torch.sin(x_val) + x_val

The estimator computes all query--key distances, applies a Gaussian kernel
of bandwidth $\sigma$, normalizes over the keys, and takes the weighted sum
of the labels. The normalized kernel matrix is the matrix of attention
weights, with one row per query, so we return it as well.

In [ ]:
def nadaraya_watson(x_train, y_train, x_val, sigma):
    dists = x_val[:, None] - x_train[None, :]  # (queries, keys)
    k = d2l.exp(-dists**2 / (2 * sigma**2))
    attention_w = k / k.sum(1, keepdims=True)  # Normalize over the keys
    return attention_w @ y_train, attention_w

How much does the bandwidth matter? We compare three bandwidths, a factor
of four or five apart, against the noise-free function; the dots are the
training data.

In [ ]:
sigmas = (0.1, 0.5, 2.0)
estimates = [nadaraya_watson(x_train, y_train, x_val, s)[0] for s in sigmas]
d2l.plot([x_val] * 4 + [x_train], estimates + [y_val, y_train], 'x', 'y',
         legend=[rf'$\sigma$ = {s:g}' for s in sigmas] + ['truth'],
         fmts=('-', 'm--', 'g-.', 'k:', 'C7o'), ylim=(-2.5, 9),
         figsize=(5, 3))

The bandwidth decides the fit. With $\sigma = 0.1$ the estimate chases the
noise of individual observations. With $\sigma = 2$ it averages over most
of the data and flattens toward the global mean, missing both the bump and
the dip of the curve. The middle bandwidth, $\sigma = 0.5$, follows the
noise-free function closely except near the ends of the interval, where
all its neighbors lie on one side of the query and pull the estimate
toward them. We keep the kernel Gaussian and vary only $\sigma$; the
other shapes of the figure can be rescaled in the
same way.

## Attention Weights as a Diagnostic

Because the weights are nonnegative and sum to one, a row of them reads as
a distribution over keys. It tells us which stored pairs a query drew on.
Plotting the weight matrix over (query, key) pairs is therefore the
standard first look at any attention layer, and we will do it throughout
the chapter with `d2l.show_heatmaps`. The helper takes an array of shape
(rows of panels, columns of panels, queries, keys) and draws a grid of
heatmaps.

The narrow and the wide kernel put weights of very different sizes on
their keys, so we give each map its own color scale (`cbar='each'`). The
queries are the test inputs and the keys are the training inputs, both in
increasing order.

In [ ]:
w_narrow, w_wide = (nadaraya_watson(x_train, y_train, x_val, s)[1]
                    for s in (0.1, 2.0))
d2l.show_heatmaps(d2l.stack([w_narrow, w_wide])[None],
                  xlabel='Training inputs (keys)',
                  ylabel='Test inputs (queries)',
                  titles=[r'$\sigma$ = 0.1', r'$\sigma$ = 2'],
                  figsize=(7, 3), cbar='each')

With the narrow kernel, each query puts nearly all of its weight on the
few keys nearest to it, so the map is a thin diagonal band and the
estimate copies a few noisy labels. The wide kernel spreads its weight
out: its largest weight is a few hundredths, spread over a broad band of
keys, so neighboring queries see nearly the same average. In both maps the
diagonal appears only because queries and keys are sorted inputs on the
same axis. In a trained network, such a heatmap is a hypothesis about what
the layer does, not a proof. A large weight on a key whose value the next
layer ignores explains nothing. that section
returns to this question with tools that test such hypotheses.

A heatmap also shows what Nadaraya--Watson regression cannot do. Its
kernel fixes the shape, the bandwidth, and the space in which distances
are measured, so every query uses the same prescribed notion of
similarity. A query cannot ask for the key three positions back, or for
the key with the same grammatical role, because neither is a matter of
distance in the input space. A learned attention layer instead computes
queries and keys from the data with trainable maps, so that the weights
they induce serve the task. The next section develops the scoring function
that makes this practical.

## Summary

Attention pooling the equation answers a query with a
weighted sum of stored values, where each weight measures how well the
query matches the value's key. It reads every stored pair directly, so it
needs no fixed-size summary of its input, and its parameters do not depend
on how many pairs there are. The softmax turns any scoring function into
nonnegative weights that sum to one; exact lookup and average pooling are
the two extremes. Nadaraya--Watson regression is attention with a fixed
kernel, and its bandwidth trades noise against bias. Heatmaps of the
weights show which keys each query uses, and learned attention replaces the
fixed kernel by learned queries and keys.

## Exercises

1. **An average of two values.** Let the keys $\mathbf{k}_1, \ldots,
   \mathbf{k}_m$ be orthonormal and let the score be $a(\mathbf{q},
   \mathbf{k}) = \mathbf{q}^\top \mathbf{k}$ with the weights
   of the equation.
    1. For the query $\mathbf{q} = c\,(\mathbf{k}_a + \mathbf{k}_b)$ with
       $c > 0$, compute all $m$ weights and show that the output tends to
       $\tfrac{1}{2}(\mathbf{v}_a + \mathbf{v}_b)$ as $c \to \infty$.
    1. How large must $c$ be for the two selected keys to receive at least
       99% of the total weight when $m = 10$?
    1. Now let the keys be orthogonal with $\|\mathbf{k}_b\| =
       2\,\|\mathbf{k}_a\|$. Find a query that again averages
       $\mathbf{v}_a$ and $\mathbf{v}_b$, and state what
       $\mathbf{q} = c\,(\mathbf{k}_a + \mathbf{k}_b)$ returns instead as
       $c \to \infty$.

    *Adapted from Stanford CS224N,
    [Assignment 4](https://web.stanford.edu/class/cs224n/assignments_w25/a4.pdf),
    Problem 1(b).*

1. [code] **Effective number of keys.** For the weights $\alpha_i(q)$ of
   one query, define the effective number of keys
   $n_{\textrm{eff}}(q) = 1 / \sum_i \alpha_i(q)^2$.
    1. Let the values be $y_i = g(x_i) + \epsilon_i$ with independent noise
       of variance $s^2$, and hold the keys fixed. Show that the variance of
       the estimate the equation at $q$ is
       $s^2 / n_{\textrm{eff}}(q)$.
    1. Compute $n_{\textrm{eff}}$ at every query in `x_val` from the weights
       that `nadaraya_watson` returns, for each bandwidth in `sigmas`, and
       plot the three curves. Then redraw the noise in `y_train` 200 times
       with `x_train` fixed, and compare the standard deviation of the
       estimate at each query with $1 / \sqrt{n_{\textrm{eff}}(q)}$.

1. [code] **Queries beyond the data.** The estimate
   the equation is a convex combination of the values, and
   the inputs `x_train` lie in the interval $[0, 5)$.
    1. For distinct scalar keys and a Gaussian kernel of fixed bandwidth,
       find the limit of the estimate as $q \to \infty$. Find the same limit
       for the constant kernel.
    1. Evaluate `nadaraya_watson` with $\sigma = 0.5$ at the queries $5,
       5.5, \ldots, 10$ and plot the estimate against $y = 2\sin(x) + x$.
       Explain why no nonnegative kernel and no bandwidth let the estimate
       follow the trend of the truth.
    1. With $\sigma = 0.5$, `nadaraya_watson` returns NaN at the query
       $q = 15$. Explain why in single precision, and compute the weights
       instead as the softmax the equation of the scores
       $-(q - x_i)^2 / (2\sigma^2)$, so that the estimate is finite at
       every query and every bandwidth.

1. [code] **A learned bandwidth.** Treat $\sigma$ in `nadaraya_watson`
   as a parameter and fit it by gradient descent on `x_train` and
   `y_train`.
    1. Before running, predict the minimizer of the training error
       $\sum_i (f(x_i) - y_i)^2$, given that $y_i$ enters the computation of
       $f(x_i)$ through its own kernel weight.
    1. Run the optimization from $\sigma = 1$, report where $\sigma$ goes,
       and compare the fitted curve with the estimates for the bandwidths in
       `sigmas`.
    1. Exclude $(x_i, y_i)$ from the estimate at $x_i$ by zeroing the
       diagonal of the kernel matrix and refit. Compare the learned
       $\sigma$ with $\sigma = 0.5$, the best of the bandwidths in `sigmas`,
       and compute the mean squared error against `y_val` for both.

1. [code] **The gradient of attention.** ● Suppose that $a(\mathbf{q},
   \mathbf{k}_i) = \mathbf{q}^\top \mathbf{k}_i$ and that $\mathbf{k}_i =
   \mathbf{v}_i$ for $i = 1, \ldots, m$. Denote by $p(\mathbf{k}_i;
   \mathbf{q})$ the distribution over keys defined by the softmax
   normalization the equation.
    1. Prove that
       $$\nabla_{\mathbf{q}} \mathop{\textrm{Attention}}(\mathbf{q}, \mathcal{D}) = \textrm{Cov}_{p(\mathbf{k}_i; \mathbf{q})}[\mathbf{k}_i].$$
    1. Conclude that this Jacobian is positive semidefinite, and determine
       for which databases it is the zero matrix.
    1. Let $\mathbf{q} = c\,\mathbf{u}$ for a fixed direction $\mathbf{u}$
       for which exactly one key attains the largest score
       $\mathbf{u}^\top \mathbf{k}_i$. Find the limit of the Jacobian as
       $c \to \infty$, and state what it implies for training the query by
       gradient descent.
    1. Verify the identity and the limit with automatic differentiation on
       a random database with $m = 8$ and $d = 4$.

1. [code] **Soft lookup as retrieval.** ● Build a database of $m$ keys
   drawn uniformly from the unit sphere in $\mathbb{R}^{16}$, with value
   $\mathbf{v}_i$ the one-hot indicator of index $i$, and weight them with
   the Gaussian kernel of bandwidth $\sigma$ normalized as
   in the equation. A query $\mathbf{q} = \mathbf{k}_i +
   \boldsymbol{\epsilon}$ with $\boldsymbol{\epsilon} \sim
   \mathcal{N}(\mathbf{0}, \tau^2 \mathbf{I})$ is retrieved correctly when
   the largest entry of the output is entry $i$.
    1. Show that the retrieved index is the index of the nearest key for
       every $\sigma > 0$, so the bandwidth does not affect which index is
       retrieved.
    1. Without noise, the weight on the correct key is
       $$\alpha_i = \frac{1}{1 + \sum_{j \neq i} \exp\big(-\|\mathbf{k}_i - \mathbf{k}_j\|^2 / 2\sigma^2\big)}.$$
       Approximate $\mathbf{k}_i^\top \mathbf{k}_j \approx 0$ for random
       unit vectors, so that $\|\mathbf{k}_i - \mathbf{k}_j\|^2 \approx 2$,
       and derive the bandwidth at which $\alpha_i$ falls to $0.9$ for
       $m = 20$ and for $m = 200$. How must $\sigma$ shrink as $m$ grows?
    1. Measure $\alpha_i$ at these two bandwidths, averaged over random
       databases, and explain the direction of its discrepancy from $0.9$.
    1. Sweep $\tau$ from $0$ to $1$ at $m = 20$ and $m = 200$, with the
       bandwidth from (b) for each $m$, and plot the retrieval accuracy
       against $\tau$. Compute the weights as the softmax
       the equation of the scores
       $-\|\mathbf{q} - \mathbf{k}_j\|^2 / 2\sigma^2$. Explain why a larger
       $m$ lowers the accuracy at fixed $\tau$ although the distance from
       the query to the correct key is unchanged.

1. [extended] **A learned metric.** ● Kernel regression with a learned
   distance [@Weinberger.Tesauro.2007] replaces the fixed similarity
   of the equation by a learned one. Draw $n = 200$ inputs
   $\mathbf{x}_i$ uniformly from $[0, 5]^5$ and labels $y_i =
   2\sin(x_{i1}) + x_{i1} + \epsilon_i$ with standard Gaussian noise, so
   that only the first coordinate matters. Score a query against a key by
   $$a(\mathbf{q}, \mathbf{k}) = -\tfrac{1}{2} \|\mathbf{W}(\mathbf{q} - \mathbf{k})\|^2, \qquad \mathbf{W} = \textrm{diag}(w_1, \ldots, w_5),$$
   normalize as in the equation, and fit $\mathbf{W}$ to
   the leave-one-out error, in which the estimate at $\mathbf{x}_i$
   excludes the pair $(\mathbf{x}_i, y_i)$. Measure the test error on
   $1000$ fresh inputs against the noise-free function.
    1. Fit the isotropic kernel $w_1 = \cdots = w_5 = 1/\sigma$ by a grid
       search over $\sigma$.
    1. Starting from the isotropic solution, fit the five scales by gradient
       descent. Compare the learned scales with the relevance of each
       coordinate, and the test error with that of the isotropic kernel and
       of an oracle that uses only the first coordinate at its best
       bandwidth.
    1. Compare the leave-one-out errors of the learned and the oracle
       metrics with their test errors, and repeat the comparison at
       $n = 1000$. What does the result say about learning a similarity from
       a finite sample?
    1. A full matrix $\mathbf{W} \in \mathbb{R}^{5 \times 5}$ maps queries
       and keys by the same linear map, whereas learned attention maps them
       by separate matrices $\mathbf{W}_q$ and $\mathbf{W}_k$. State two
       properties of the score $a$ that the shared map guarantees and
       separate maps lose.

[Discussions](https://d2l.discourse.group/t/36768)